# Hybrid Search Engine

In this notebook, we combine TF-IDF and Neural Embeddings to build a hybrid search system.

This approach leverages both lexical matching and semantic understanding.

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_csv("../Data/final_tweets.csv")

print(df.shape)
df.head()

(59184, 4)


,tweet_id,text,clean_text,company
0,277379,Another great set of @Delta flights thank to #...,another great set of flights thank to tmobilew...,delta
1,2488664,@AppleSupport why is my iPhone automatically g...,why is my iphone automatically going on mute,applesupport
2,2383194,@Uber_Support Since yesterday I haven't receiv...,since yesterday i havent received any update a...,uber_support
3,1610886,"@AmazonHelp There is no email from 2 days, jus...",there is no email from days just asked to wait...,amazonhelp
4,2763258,"@SouthwestAir thanks for responding, will call...",thanks for responding will call as soon as i g...,southwestair


In [3]:
search_df = df[["tweet_id", "clean_text", "company"]].copy()

search_df.head()

,tweet_id,clean_text,company
0,277379,another great set of flights thank to tmobilew...,delta
1,2488664,why is my iphone automatically going on mute,applesupport
2,2383194,since yesterday i havent received any update a...,uber_support
3,1610886,there is no email from days just asked to wait...,amazonhelp
4,2763258,thanks for responding will call as soon as i g...,southwestair


In [4]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(search_df["clean_text"])

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (59184, 10000)


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [6]:
texts = search_df["clean_text"].tolist()

embeddings = model.encode(texts, show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)


Batches:   0%|          | 0/1850 [00:00<?, ?it/s]

Embeddings shape: (59184, 384)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def hybrid_search(query, top_k=5, alpha=0.5):
    # TF-IDF score
    query_tfidf = tfidf.transform([query])
    tfidf_scores = cosine_similarity(query_tfidf, tfidf_matrix).flatten()
    
    # Embedding score
    query_emb = model.encode([query])
    emb_scores = cosine_similarity(query_emb, embeddings).flatten()
    
    # Combine both scores
    hybrid_scores = alpha * tfidf_scores + (1 - alpha) * emb_scores
    
    # Top results
    top_indices = hybrid_scores.argsort()[::-1][:top_k]
    
    results = search_df.iloc[top_indices].copy()
    results["score"] = hybrid_scores[top_indices]
    
    return results

In [8]:
hybrid_search("flight delayed customer service", top_k=5)

,tweet_id,clean_text,company,score
2472,1799160,flight was already delayed on top of it poor c...,americanair,0.834247
30389,565503,why is flight delayed,americanair,0.715450
51365,1370386,when your flight is delayed,southwestair,0.711692
36349,2098673,my flight has been delayed hours and no custom...,americanair,0.687231
46536,2258648,been there done that with customer service thr...,americanair,0.657047


The hybrid search model combines TF-IDF and neural embedding similarity scores to improve retrieval quality. TF-IDF captures exact keyword matches, while embeddings capture semantic meaning. The hybrid approach achieves better relevance by balancing both lexical and semantic information, resulting in more accurate search results compared to individual methods.